# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 36.8418


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability


## 2. Fetch Results

In [3]:
import seml
import pandas as pd

db_name = 'daro'
#states=["FAILED"]
states = []

all_results = seml.evaluation.get_results(db_name, to_data_frame=True, states=states)
print(f"Lenght of all_results: {len(all_results)}")
print(all_results.columns)

all_results.iloc[:, 20:25].head()

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-package
s/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Lenght of all_results: 12
Index(['_id', 'config.overwrite', 'config.db_collection', 'config.batch_size',
       'config.calib_dataset_name', 'config.calib_dataset_split',
       'config.clean_cache', 'config.dataset_seq_length',
       'config.dataset_stride', 'config.device', 'config.eval_dataset_name',
       'config.eval_dataset_split', 'config.eval_metrics', 'config.model_name',
       'config.quantize_method', 'config.quantized_model_save_path',
       'config.save_quantized_model', 'config.seed', 'result.current_gpu_type',
       'result.current_gpu_total_memory', 'result.current_gpu_free_memory',
       'result.perplexity', 'result.brier_score', 'result.model_size',
       'result.quantize_runtime', 'result.fail_trace'],
      dtype='object')


,result.current_gpu_free_memory,result.perplexity,result.brier_score,result.model_size,result.quantize_runtime
0,23.896484,4.572106,0.000004,nan,nan
1,31.123047,4.989842,0.000004,7.571443758904934 GB,105.381251
2,28.445312,4.613629,0.000004,10.42031535692513 GB,106.066191
3,35.917969,4.816281,0.000004,0 B,1018.837144
4,29.294922,4.572022,0.000004,nan,128.013785


In [4]:
columns_to_remove = [
    'config.overwrite',
    'config.db_collection',
    'config.clean_cache',
    'config.device',
    'config.quantized_model_save_path',
    'config.save_quantized_model',
    'config.seed',
    'result'
]

all_results = all_results.drop(columns=columns_to_remove, errors='ignore')
all_results.columns

Index(['_id', 'config.batch_size', 'config.calib_dataset_name',
       'config.calib_dataset_split', 'config.dataset_seq_length',
       'config.dataset_stride', 'config.eval_dataset_name',
       'config.eval_dataset_split', 'config.eval_metrics', 'config.model_name',
       'config.quantize_method', 'result.current_gpu_type',
       'result.current_gpu_total_memory', 'result.current_gpu_free_memory',
       'result.perplexity', 'result.brier_score', 'result.model_size',
       'result.quantize_runtime', 'result.fail_trace'],
      dtype='object')

In [5]:
from src.algorithms.quantization import QUANT_CONFIGS

# Define a function to label datasets explicitly
def label_dataset(row):
    return f"Calib: {row['config.calib_dataset_name']} ({row['config.calib_dataset_split']}) | Eval: {row['config.eval_dataset_name']} ({row['config.eval_dataset_split']})"

# Adding new columns to check if splits and datasets are the same
all_results['split_equals'] = all_results['config.calib_dataset_split'] == all_results['config.eval_dataset_split']
all_results['dataset_equals'] = all_results['config.calib_dataset_name'] == all_results['config.eval_dataset_name']
all_results['dataset_label'] = all_results.apply(label_dataset, axis=1)
all_results.rename(columns={'config.quantize_method': 'config.quantize_method_name'}, inplace=True)

# Function to extract quantize_method and n_bits
def extract_quantize_method_and_bits(quantize_method):
    config = QUANT_CONFIGS[quantize_method]
    return config['quantize_method'], config['n_bits']

# Apply the function to create new columns
all_results[['config.quantize_method', 'config.n_bits']] = all_results['config.quantize_method_name'].apply(
    lambda x: pd.Series(extract_quantize_method_and_bits(x))
)

# Rename the column
all_results.drop(columns=['config.quantize_method_name'], inplace=True)
all_results.head()

,_id,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.dataset_seq_length,config.dataset_stride,config.eval_dataset_name,config.eval_dataset_split,config.eval_metrics,config.model_name,...,result.perplexity,result.brier_score,result.model_size,result.quantize_runtime,result.fail_trace,split_equals,dataset_equals,dataset_label,config.quantize_method,config.n_bits
0,1,1,WikiText,validation,1024,1024,OpenAssistant,test,"[perplexity, brier_score, quantize_runtime, mo...",Llama-3-8B,...,4.572106,0.000004,nan,nan,<function get_results at 0x7fee6bc885e0>,False,False,Calib: WikiText (validation) | Eval: OpenAssis...,NONE,16
1,2,1,WikiText,validation,1024,1024,OpenAssistant,test,"[perplexity, brier_score, quantize_runtime, mo...",Llama-3-8B,...,4.989842,0.000004,7.571443758904934 GB,105.381251,<function get_results at 0x7fee6bc885e0>,False,False,Calib: WikiText (validation) | Eval: OpenAssis...,BNB,4
2,3,1,WikiText,validation,1024,1024,OpenAssistant,test,"[perplexity, brier_score, quantize_runtime, mo...",Llama-3-8B,...,4.613629,0.000004,10.42031535692513 GB,106.066191,<function get_results at 0x7fee6bc885e0>,False,False,Calib: WikiText (validation) | Eval: OpenAssis...,BNB,8
3,4,1,WikiText,validation,1024,1024,OpenAssistant,test,"[perplexity, brier_score, quantize_runtime, mo...",Llama-3-8B,...,4.816281,0.000004,0 B,1018.837144,<function get_results at 0x7fee6bc885e0>,False,False,Calib: WikiText (validation) | Eval: OpenAssis...,AWQ,4
4,5,1,WikiText,validation,1024,1024,OpenAssistant,test,"[perplexity, brier_score, quantize_runtime, mo...",Llama-3-8B,...,4.572022,0.000004,nan,128.013785,<function get_results at 0x7fee6bc885e0>,False,False,Calib: WikiText (validation) | Eval: OpenAssis...,HQQ,8


In [8]:
all_results.columns

Index(['_id', 'config.batch_size', 'config.calib_dataset_name',
       'config.calib_dataset_split', 'config.dataset_seq_length',
       'config.dataset_stride', 'config.eval_dataset_name',
       'config.eval_dataset_split', 'config.eval_metrics', 'config.model_name',
       'result.current_gpu_type', 'result.current_gpu_total_memory',
       'result.current_gpu_free_memory', 'result.perplexity',
       'result.brier_score', 'result.model_size', 'result.quantize_runtime',
       'result.fail_trace', 'split_equals', 'dataset_equals', 'dataset_label',
       'config.quantize_method', 'config.n_bits'],
      dtype='object')

## 3. Plot results

In [12]:
all_results["config.model_name"]

0     Llama-3-8B
1     Llama-3-8B
2     Llama-3-8B
3     Llama-3-8B
4     Llama-3-8B
5     Llama-3-8B
6      TinyLlama
7      TinyLlama
8      TinyLlama
9      TinyLlama
10     TinyLlama
11     TinyLlama
Name: config.model_name, dtype: object

In [15]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Assuming the dataframe `all_results` is already loaded and filtered

# Plot 1: Perplexity by Quantization Method and Number of Bits
fig1 = px.box(all_results.loc[all_results["config.model_name"] == "TinyLlama"], 
              x='config.quantize_method', 
              y='result.perplexity', 
              color='config.n_bits', 
              title='Perplexity for TinyLlama',
              labels={'config.quantize_method': 'Quantization Method', 'result.perplexity': 'Perplexity'},
              points='all')
fig1.show()

# Plot 1.1: Perplexity by Quantization Method and Number of Bits
fig1_1 = px.box(all_results.loc[all_results["config.model_name"] == "Llama-3-8B"], 
              x='config.quantize_method', 
              y='result.perplexity', 
              color='config.n_bits', 
              title='Perplexity for Llama-3-8B',
              labels={'config.quantize_method': 'Quantization Method', 'result.perplexity': 'Perplexity'},
              points='all')
fig1_1.show()

# Plot 2: Brier Score by Quantization Method and Number of Bits
fig2 = px.box(all_results.loc[all_results["config.model_name"] == "TinyLlama"], 
              x='config.quantize_method', 
              y='result.brier_score', 
              color='config.n_bits', 
              title='Brier Score for TinyLlama',
              labels={'config.quantize_method': 'Quantization Method', 'result.brier_score': 'Brier Score'},
              points='all')
fig2.show()

# Plot 2.2: Brier Score by Quantization Method and Number of Bits
fig2_2 = px.box(all_results.loc[all_results["config.model_name"] == "Llama-3-8B"], 
              x='config.quantize_method', 
              y='result.brier_score', 
              color='config.n_bits', 
              title='Brier Score for Llama-3-8B',
              labels={'config.quantize_method': 'Quantization Method', 'result.brier_score': 'Brier Score'},
              points='all')
fig2_2.show()

# Plot 3: Perplexity by Calibration and Evaluation Dataset
fig3 = px.scatter(all_results, 
                  x='config.calib_dataset_name', 
                  y='result.perplexity', 
                  color='config.eval_dataset_name', 
                  symbol='config.eval_dataset_name',
                  title='Perplexity by Calibration and Evaluation Dataset',
                  labels={'config.calib_dataset_name': 'Calibration Dataset', 'result.perplexity': 'Perplexity'})
fig3.show()

# Plot 4: Brier Score by Calibration and Evaluation Dataset
fig4 = px.scatter(all_results, 
                  x='config.calib_dataset_name', 
                  y='result.brier_score', 
                  color='config.eval_dataset_name', 
                  symbol='config.eval_dataset_name',
                  title='Brier Score by Calibration and Evaluation Dataset',
                  labels={'config.calib_dataset_name': 'Calibration Dataset', 'result.brier_score': 'Brier Score'})
fig4.show()

# Plot 5: Memory Usage by Quantization Method and Number of Bits
fig5 = px.box(all_results, 
              x='config.quantize_method', 
              y='result.current_gpu_total_memory', 
              color='config.n_bits', 
              title='Memory Usage by Quantization Method and Number of Bits',
              labels={'config.quantize_method': 'Quantization Method', 'result.current_gpu_total_memory': 'Memory Usage (MB)'},
              points='all')
fig5.show()

# Plot 6: Perplexity when Calibration and Evaluation Datasets are Same vs Different
fig6 = px.box(all_results, 
              x='dataset_equals', 
              y='result.perplexity', 
              title='Perplexity when Calibration and Evaluation Datasets are Same vs Different',
              labels={'dataset_equals': 'Calibration Equals Evaluation Dataset', 'result.perplexity': 'Perplexity'},
              points='all')
fig6.show()

# Plot 7: Brier Score when Calibration and Evaluation Datasets are Same vs Different
fig7 = px.box(all_results, 
              x='dataset_equals', 
              y='result.brier_score', 
              title='Brier Score when Calibration and Evaluation Datasets are Same vs Different',
              labels={'dataset_equals': 'Calibration Equals Evaluation Dataset', 'result.brier_score': 'Brier Score'},
              points='all')
fig7.show()

# Plot 8: Perplexity by Calibration and Evaluation Data Split
fig8 = px.scatter(all_results, 
                  x='config.calib_dataset_split', 
                  y='result.perplexity', 
                  color='config.eval_dataset_split', 
                  symbol='dataset_equals',
                  title='Perplexity by Calibration and Evaluation Data Split',
                  labels={'config.calib_dataset_split': 'Calibration Data Split', 'result.perplexity': 'Perplexity'},
                  hover_data=['dataset_label'])
fig8.show()

# Plot 9: Brier Score by Calibration and Evaluation Data Split
fig9 = px.scatter(all_results, 
                  x='config.calib_dataset_split', 
                  y='result.brier_score', 
                  color='config.eval_dataset_split', 
                  symbol='dataset_equals',
                  title='Brier Score by Calibration and Evaluation Data Split',
                  labels={'config.calib_dataset_split': 'Calibration Data Split', 'result.brier_score': 'Brier Score'},
                  hover_data=['dataset_label'])
fig9.show()

# Plot 10: Perplexity when Calibration and Evaluation Data Split are Same vs Different
fig10 = px.box(all_results, 
               x='split_equals', 
               y='result.perplexity', 
               color='dataset_equals',
               title='Perplexity when Calibration and Evaluation Data Split are Same vs Different',
               labels={'split_equals': 'Calibration Equals Evaluation Split', 'result.perplexity': 'Perplexity'},
               points='all')
fig10.show()

# Plot 11: Brier Score when Calibration and Evaluation Data Split are Same vs Different
fig11 = px.box(all_results, 
               x='split_equals', 
               y='result.brier_score', 
               color='dataset_equals',
               title='Brier Score when Calibration and Evaluation Data Split are Same vs Different',
               labels={'split_equals': 'Calibration Equals Evaluation Split', 'result.brier_score': 'Brier Score'},
               points='all')
fig11.show()

# Plot 12: Perplexity by Data Split and Dataset Combination
fig12 = px.box(all_results, 
               x='split_equals', 
               y='result.perplexity', 
               color='dataset_equals',
               title='Perplexity by Data Split and Dataset Combination',
               labels={'split_equals': 'Calibration Equals Evaluation Split', 'result.perplexity': 'Perplexity'},
               facet_col='dataset_equals',
               points='all')
fig12.show()

# Plot 13: Brier Score by Data Split and Dataset Combination
fig13 = px.box(all_results, 
               x='split_equals', 
               y='result.brier_score', 
               color='dataset_equals',
               title='Brier Score by Data Split and Dataset Combination',
               labels={'split_equals': 'Calibration Equals Evaluation Split', 'result.brier_score': 'Brier Score'},
               facet_col='dataset_equals',
               points='all')
fig13.show()

In [13]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from fpdf import FPDF
import os

# Create a list of plots and their descriptions
plots = [
    (fig1, "Plot 1: Perplexity by Quantization Method and Number of Bits"),
    (fig2, "Plot 2: Brier Score by Quantization Method and Number of Bits"),
    (fig3, "Plot 3: Perplexity by Calibration and Evaluation Dataset"),
    (fig4, "Plot 4: Brier Score by Calibration and Evaluation Dataset"),
    (fig5, "Plot 5: Memory Usage by Quantization Method and Number of Bits"),
    (fig6, "Plot 6: Perplexity when Calibration and Evaluation Datasets are Same vs Different"),
    (fig7, "Plot 7: Brier Score when Calibration and Evaluation Datasets are Same vs Different"),
    (fig8, "Plot 8: Perplexity by Calibration and Evaluation Data Split"),
    (fig9, "Plot 9: Brier Score by Calibration and Evaluation Data Split"),
    (fig10, "Plot 10: Perplexity when Calibration and Evaluation Data Split are Same vs Different"),
    (fig11, "Plot 11: Brier Score when Calibration and Evaluation Data Split are Same vs Different"),
    (fig12, "Plot 12: Perplexity by Data Split and Dataset Combination"),
    (fig13, "Plot 13: Brier Score by Data Split and Dataset Combination")
]

plot_save_path = "plots"
# Save plots as images
for i, (fig, desc) in enumerate(plots):
    fig.write_image(os.path.join(plot_save_path, f"plot_{i}.png"))

# Create a PDF document
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

# Add plots and descriptions to the PDF
for i, (fig, desc) in enumerate(plots):
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 10, desc)
    pdf.ln(10)
    pdf.image(os.path.join(plot_save_path, f"plot_{i}.png"), w=pdf.w - 30)

# Save the PDF
pdf.output(os.path.join(plot_save_path, "results_summary.pdf"), "F")

print("PDF generated and saved as 'results_summary.pdf'")

PDF generated and saved as 'results_summary.pdf'


In [12]:
!

LICENSE    __init__.py	plots		  results_summary.pdf  src
README.md  notebooks	pyproject.toml	  scripts
TinyLlama  outfiles	requirements.txt  setup.py


In [15]:
model_path = "ssss"
if not os.path.exists(model_path):
  print("nan")

nan
